# 🚀 Entrenamiento LSTM de Clientes en Kaggle

**GPU disponible:** Tesla P100 o T4 (gratis)
**Tiempo estimado:** 2-4 horas
**Costo:** $0

---

## ⚙️ IMPORTANTE: Activar GPU
1. Click derecho en este notebook → **Settings**
2. En **Accelerator**, selecciona **GPU T4 x2** o **GPU P100**
3. Click **Save**

---

## 1️⃣ Verificar GPU

In [ ]:
import tensorflow as tf

print("="*80)
print("CONFIGURACIÓN DE KAGGLE")
print("="*80)
print(f"TensorFlow version: {tf.__version__}")
print(f"GPU disponible: {tf.config.list_physical_devices('GPU')}")

if len(tf.config.list_physical_devices('GPU')) > 0:
    print("\n✅ GPU ACTIVADA - El entrenamiento será RÁPIDO")
    gpu_info = tf.config.experimental.get_device_details(tf.config.list_physical_devices('GPU')[0])
    print(f"GPU: {gpu_info}")
else:
    print("\n⚠️ GPU NO DETECTADA - Ve a Settings → Accelerator → GPU")

## 2️⃣ Instalar dependencias

In [ ]:
!pip install -q openpyxl seaborn
print("\n✅ Dependencias instaladas")

## 3️⃣ Subir archivos

**Opción A:** Usa Kaggle Datasets (recomendado)
1. Ve a https://www.kaggle.com/datasets
2. Click **New Dataset**
3. Sube `train_all_customers_temporal.py` y `online_retail_2.xlsx`
4. Haz el dataset público o privado
5. En este notebook: **Add Data** → busca tu dataset

**Opción B:** Sube archivos directamente aquí

In [ ]:
# Si subiste como dataset, los archivos estarán en /kaggle/input/
import os

# Listar archivos disponibles
!ls -lh /kaggle/input/

# Si subiste directamente, usa:
# from google.colab import files (NO funciona en Kaggle)
# En Kaggle debes usar Datasets

## 4️⃣ Copiar archivos y crear estructura

In [ ]:
# Crear estructura de directorios
!mkdir -p data/processed
!mkdir -p models/temporal/customer/short
!mkdir -p models/temporal/customer/medium
!mkdir -p models/temporal/customer/long

# Copiar archivos desde el dataset (ajusta la ruta según tu dataset)
# Ejemplo: si tu dataset se llama "lstm-customer-training"
!cp /kaggle/input/lstm-customer-training/online_retail_2.xlsx data/processed/
!cp /kaggle/input/lstm-customer-training/train_all_customers_temporal.py .

print("✅ Estructura creada")
!ls -R

## 5️⃣ Importar script de entrenamiento

In [ ]:
import sys
sys.path.append('.')

try:
    from train_all_customers_temporal import CustomerTemporalAnalyzer, TemporalConfig
    print("✅ Script importado correctamente")
except Exception as e:
    print(f"❌ ERROR: {e}")

## 6️⃣ Configuración de entrenamiento

**Selecciona qué horizontes entrenar:**

In [ ]:
# Opción: Entrenar solo los que faltan (MEDIUM + LONG)
horizons_to_train = [
    TemporalConfig.MEDIUM,
    TemporalConfig.LONG
]

# O entrenar todos:
# horizons_to_train = [TemporalConfig.SHORT, TemporalConfig.MEDIUM, TemporalConfig.LONG]

print(f"✅ Configurado para entrenar {len(horizons_to_train)} horizonte(s)")

## 7️⃣ Inicializar y preparar datos

In [ ]:
import warnings
warnings.filterwarnings('ignore')
from datetime import datetime

start_time = datetime.now()
print(f"⏰ Inicio: {start_time.strftime('%Y-%m-%d %H:%M:%S')}\n")

# Inicializar
analyzer = CustomerTemporalAnalyzer(
    data_path='data/processed/online_retail_2.xlsx',
    output_dir='models/temporal/customer'
)

# Preparar datos (común para todos)
print("="*70)
print("FASE 1: Preparación de datos")
print("="*70)
analyzer.load_and_preprocess_data()
analyzer.calculate_rfm_metrics()
analyzer.generate_customer_sequences(min_transactions=5)

print("\n✅ Datos preparados")

## 8️⃣ ENTRENAR MODELOS

**Esta celda tomará 2-4 horas. Puedes cerrar la pestaña y volver después.**

In [ ]:
import time

print(f"\n{'='*70}")
print(f"FASE 2: Entrenamiento de {len(horizons_to_train)} horizonte(s)")
print(f"{'='*70}")

results = {}

for i, horizon_config in enumerate(horizons_to_train, 1):
    print(f"\n\n{'█'*70}")
    print(f"HORIZONTE {i}/{len(horizons_to_train)}: {horizon_config['name'].upper()}")
    print(f"{'█'*70}")
    
    horizon_start = time.time()
    
    try:
        model, history, metrics = analyzer.train_horizon_model(horizon_config)
        
        horizon_duration = (time.time() - horizon_start) / 60
        
        results[horizon_config['name']] = {
            'status': 'SUCCESS',
            'metrics': metrics,
            'duration_minutes': horizon_duration
        }
        
        print(f"\n✅ {horizon_config['name'].upper()} completado en {horizon_duration:.1f} minutos")
        
    except Exception as e:
        print(f"\n❌ Error entrenando {horizon_config['name']}: {e}")
        results[horizon_config['name']] = {
            'status': 'FAILED',
            'error': str(e)
        }

# Resumen final
end_time = datetime.now()
total_duration = (end_time - start_time).total_seconds() / 60

print(f"\n\n{'═'*70}")
print("RESUMEN FINAL DE ENTRENAMIENTO")
print(f"{'═'*70}\n")

for horizon, result in results.items():
    status_icon = "✅" if result['status'] == 'SUCCESS' else "❌"
    print(f"{status_icon} {horizon.upper()}: {result['status']}")
    
    if result['status'] == 'SUCCESS':
        metrics = result['metrics']
        print(f"   Accuracy: {metrics['purchase_prob_accuracy']*100:.2f}%")
        print(f"   AUC: {metrics['purchase_prob_auc']:.4f}")
        print(f"   Days MAE: {metrics['days_mae']:.2f}")
        print(f"   Value MAE: ${metrics['value_mae']:.2f}")
        print(f"   Tiempo: {result['duration_minutes']:.1f} min")
    print()

print(f"⏰ Tiempo total: {total_duration:.1f} minutos ({total_duration/60:.1f} horas)")
print(f"\n💾 Modelos guardados en: {analyzer.output_dir}/")
print("\n" + "="*70)
print("✅ ENTRENAMIENTO COMPLETADO")
print("="*70)

## 9️⃣ Descargar modelos

**Kaggle permite descargar archivos directamente desde el notebook.**

In [ ]:
# Comprimir modelos
!cd models/temporal && zip -r customer_models_kaggle.zip customer/

print("✅ Modelos comprimidos")
print("\n📥 Para descargar:")
print("   1. Click en 'Output' (arriba a la derecha)")
print("   2. Descarga 'customer_models_kaggle.zip'")
print("   3. Descomprime en: E:\\Codigos\\Proyecto Final\\models\\temporal\\")

!ls -lh models/temporal/*.zip

## 🔟 Verificar archivos generados

In [ ]:
print("📊 ARCHIVOS GENERADOS:\n")

for horizon in ['short', 'medium', 'long']:
    print(f"\n{horizon.upper()}:")
    !ls -lh models/temporal/customer/{horizon}/ 2>/dev/null || echo "  (no entrenado)"